In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/processed/feature_engineered.csv"
)

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (64283, 66)
           Patient            timestamp  glucose_value meal_type  meal_carbs  \
0  540-ws-training  2027-05-19 11:36:29             76      none         0.0   
1  540-ws-training  2027-05-19 11:41:29             72      none         0.0   
2  540-ws-training  2027-05-19 11:46:29             68      none         0.0   
3  540-ws-training  2027-05-19 11:51:29             65      none         0.0   
4  540-ws-training  2027-05-19 11:56:29             63      none         0.0   

  bolus_type  bolus_dose  basal_value  temp_basal_value exercise_type  ...  \
0     normal         0.8         0.95               0.0          none  ...   
1     normal         0.8         0.95               0.0          none  ...   
2     normal         0.8         0.95               0.0          none  ...   
3     normal         0.8         0.95               0.0          none  ...   
4     normal         0.8         0.95               0.0          none  ...   

   sleep_change gsr_rol

C:\Users\Naveen kumar\AppData\Local\Temp\ipykernel_15920\3682199530.py:4: DtypeWarning: Columns (0: exercise_intensity) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


In [2]:
print("Target exists:", "glucose_30min_ahead" in df.columns)

Target exists: True


In [3]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

df = df.sort_values(
    ["Patient", "timestamp"]
).reset_index(drop=True)

print(df[["Patient", "timestamp", "glucose_30min_ahead"]].head())

           Patient           timestamp  glucose_30min_ahead
0  540-ws-training 2027-05-19 11:36:29                 71.0
1  540-ws-training 2027-05-19 11:41:29                 78.0
2  540-ws-training 2027-05-19 11:46:29                 90.0
3  540-ws-training 2027-05-19 11:51:29                 99.0
4  540-ws-training 2027-05-19 11:56:29                110.0


In [4]:
drop_cols = [
    "Patient",
    "timestamp",
    "glucose_30min_ahead",
    "meal_type",
    "bolus_type",
    "exercise_type",
    "exercise_intensity",
    "stressor_type",
    "stressor_description",
    "illness_type",
    "illness_description",
    "time_period"
]

X = df.drop(
    columns=drop_cols,
    errors="ignore"
)

y = df["glucose_30min_ahead"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (64283, 54)
y shape: (64283,)


In [5]:
# Make sure data is sorted
df = df.sort_values(
    ["Patient", "timestamp"]
).reset_index(drop=True)

# Create train/test masks patient-wise
train_parts = []
test_parts = []

for patient, patient_df in df.groupby("Patient"):
    
    split_index = int(len(patient_df) * 0.8)
    
    train_parts.append(patient_df.iloc[:split_index])
    test_parts.append(patient_df.iloc[split_index:])

train_df = pd.concat(train_parts).reset_index(drop=True)
test_df = pd.concat(test_parts).reset_index(drop=True)

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

Training shape: (51424, 66)
Testing shape: (12859, 66)


In [6]:
print("\nTraining date range:")
print(train_df["timestamp"].min())
print(train_df["timestamp"].max())

print("\nTesting date range:")
print(test_df["timestamp"].min())
print(test_df["timestamp"].max())


Training date range:
2025-04-16 11:17:05
2027-06-23 21:09:15

Testing date range:
2025-05-17 06:36:54
2027-07-03 23:26:44


In [7]:
print("\nPatient-wise split:")
for patient in df["Patient"].unique():
    
    train_patient = train_df[
        train_df["Patient"] == patient
    ]
    
    test_patient = test_df[
        test_df["Patient"] == patient
    ]
    
    print(
        patient,
        "| Train:", len(train_patient),
        "| Test:", len(test_patient)
    )


Patient-wise split:
540-ws-training | Train: 9431 | Test: 2358
544-ws-training | Train: 8392 | Test: 2099
552-ws-training | Train: 7092 | Test: 1774
567-ws-training | Train: 8430 | Test: 2108
584-ws-training | Train: 9487 | Test: 2372
596-ws-training | Train: 8592 | Test: 2148


In [8]:
target = "glucose_30min_ahead"

X_train = train_df.drop(
    columns=drop_cols,
    errors="ignore"
)

y_train = train_df[target]

X_test = test_df.drop(
    columns=drop_cols,
    errors="ignore"
)

y_test = test_df[target]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (51424, 54)
y_train: (51424,)
X_test: (12859, 54)
y_test: (12859,)


In [9]:
print("Non-numeric training columns:")

print(
    X_train.select_dtypes(
        exclude=np.number
    ).columns.tolist()
)

Non-numeric training columns:
['work_intensity']


In [10]:
print("\nNumber of features:", X_train.shape[1])
print("\nData types:")
print(X_train.dtypes)


Number of features: 54

Data types:
glucose_value                       int64
meal_carbs                        float64
bolus_dose                        float64
basal_value                       float64
temp_basal_value                  float64
exercise_duration                 float64
gsr                               float64
skin_temperature                  float64
acceleration                      float64
is_sleeping                         int64
sleep_quality                     float64
is_working                          int64
work_intensity                        str
hypo_event                          int64
illness_event                       int64
fingerstick_glucose               float64
hour                                int64
day_of_week                         int64
is_weekend                          int64
glucose_prev_5min                 float64
glucose_prev_10min                float64
glucose_prev_15min                float64
glucose_prev_30min                float

In [11]:
print("Work intensity values:")
print(X_train["work_intensity"].value_counts(dropna=False))

Work intensity values:
work_intensity
none    47448
5.0      1631
3.0      1142
2.0       690
4.0       413
6.0       100
Name: count, dtype: int64


In [12]:
X_train = pd.get_dummies(
    X_train,
    columns=["work_intensity"],
    dtype=float
)

X_test = pd.get_dummies(
    X_test,
    columns=["work_intensity"],
    dtype=float
)

# Make sure train and test have exactly the same columns
X_train, X_test = X_train.align(
    X_test,
    join="left",
    axis=1,
    fill_value=0
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nRemaining non-numeric columns:")
print(
    X_train.select_dtypes(
        exclude=np.number
    ).columns.tolist()
)

X_train shape: (51424, 59)
X_test shape: (12859, 59)

Remaining non-numeric columns:
[]


In [13]:
print("Missing values in X_train:")
print(X_train.isna().sum().sum())

print("\nMissing values in X_test:")
print(X_test.isna().sum().sum())

Missing values in X_train:
38961

Missing values in X_test:
8716


In [14]:
X_train_np = X_train.astype("float32").values
X_test_np = X_test.astype("float32").values

y_train_np = y_train.astype("float32").values
y_test_np = y_test.astype("float32").values

print("X_train:", X_train_np.shape)
print("X_test:", X_test_np.shape)
print("y_train:", y_train_np.shape)
print("y_test:", y_test_np.shape)

X_train: (51424, 59)
X_test: (12859, 59)
y_train: (51424,)
y_test: (12859,)


In [ ]:
import tensorflow as tf
import keras

normalizer = keras.layers.Normalization()

# IMPORTANT:
# Adapt only using training data
normalizer.adapt(X_train_np)

print("Normalization layer ready.")

AttributeError: module 'tensorflow' has no attribute 'keras'